In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import warnings
warnings.filterwarnings("ignore")
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

# —— 1. 读取数据 & 特征工程 ——  
# —— 1. Load data & feature engineering ——  
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test  = pd.read_csv('/kaggle/input/titanic/test.csv')

def add_features(df):
    # a) 提取称谓 Title / Extract Title from Name
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
    # b) 家庭规模 FamilySize = SibSp + Parch + 1
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    # c) 船舱首字母 CabinLetter (填缺失为 U)  
    # c) CabinLetter (fill missing with 'U')
    df['CabinLetter'] = df['Cabin'].fillna('U').str[0]
    return df

# 应用特征工程 / Apply to both sets
train = add_features(train)
test  = add_features(test)

# —— 2. 定义特征列 ——  
# —— 2. Define feature columns ——  
num_cols = ['Age','Fare','FamilySize','SibSp','Parch']
cat_cols = ['Pclass','Sex','Embarked','Title','CabinLetter']

# —— 3. 构建预处理流水线 ——  
# —— 3. Build preprocessing pipeline ——  
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # 数值中位数填充 / median imputation
    ('scaler', StandardScaler())                     # 标准化 / standard scaling
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),  # 类别常数填充 / constant fill
    ('onehot', OneHotEncoder(handle_unknown='ignore'))                      # One‑Hot 编码
])
preproc = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# —— 4. 定义 RandomForest & 随机搜索空间 ——  
# —— 4. Define RF and hyperparameter search space ——  
rf = RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True, class_weight='balanced')

param_dist = {
    'clf__n_estimators': np.arange(100, 601, 50),        # 树的数量 / number of trees
    'clf__max_depth':    [None] + list(np.arange(5, 21, 5)),  # 最大深度 / max depth
    'clf__min_samples_split': np.arange(2, 11),          # 最小分裂样本数 / min samples split
    'clf__min_samples_leaf':  np.arange(1, 5),           # 叶节点最小样本 / min samples leaf
    'clf__max_features': ['auto','sqrt','log2']          # 最大特征数 / max features
}

pipeline = Pipeline([
    ('preproc', preproc),
    ('clf', rf)
])

# —— 5. RandomizedSearchCV 超参数搜索 ——  
# —— 5. RandomizedSearchCV hyperparam tuning ——  
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=30,                     # 随机采 30 组 / 30 random samples
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
X = train.drop(columns=['Survived','PassengerId','Name','Ticket','Cabin'])
y = train['Survived']
search.fit(X, y)

print("最优参数／Best params:", search.best_params_)
print("CV 得分／Best CV accuracy:", search.best_score_)

# —— 6. 用最优模型在全量数据拟合 & 生成 submission.csv ——  
# —— 6. Fit best model on all train & make submission.csv ——  
best_model = search.best_estimator_
best_model.fit(X, y)

# 准备测试集 / Prepare test features
X_test = test.drop(columns=['PassengerId','Name','Ticket','Cabin'])
preds  = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': preds
})
submission.to_csv('submission.csv', index=False)
print("已生成 submission.csv / Created submission.csv")


Fitting 5 folds for each of 30 candidates, totalling 150 fits


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomForestClassifiers and ExtraTreesClassifiers.
  warn(
/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_forest.py:424: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features='sqrt'` or remove this parameter as it is also the default value for RandomFor

最优参数／Best params: {'clf__n_estimators': 600, 'clf__min_samples_split': 2, 'clf__min_samples_leaf': 3, 'clf__max_features': 'auto', 'clf__max_depth': 15}
CV 得分／Best CV accuracy: 0.8316427091833531
已生成 submission.csv / Created submission.csv
